RAG Based chatbot insurance
In this section, I identify and load all the insurance PDF documents
that will be used for the RAG chatbot.

In [2]:
import os

pdf_folder = "data"

pdf_files = os.listdir(pdf_folder)

print("Total files:", len(pdf_files))
print("Files:")
for file in pdf_files:
    print(file)

Total files: 9
Files:
.ipynb_checkpoints
care--policy-usage-guide.pdf
care-freedom-(health-insurance-product)---brochure.pdf
care-freedom-(health-insurance-product)---prospectus-cum-sales-literature.pdf
care-opd---prospectus-cum-sales-literature.pdf
care-plus-(health-insurance-product)---brochure.pdf
care-plus--(health-insurance-product)---prospectus-cum-sales-literature.pdf
care-supreme----policy-usage-guide.pdf
Untitled.ipynb


In [7]:
!pip install PyMuPDF

In [8]:
import fitz

print("PyMuPDF installed successfully!")

PyMuPDF installed successfully!


I use PyMuPDF (`fitz`) to open the PDF files and extract text
from each page.

This helps convert the PDF documents into text that can later
be processed and indexed for retrieval.

In [5]:
import fitz
import os

pdf_path = os.path.join(pdf_folder, pdf_files[0])

doc = fitz.open(pdf_path)

print("PDF:", pdf_files[0])
print("Number of pages:", len(doc))

FileDataError: 'data\.ipynb_checkpoints' is no file

In [8]:
text = ""

for page in doc:
    text += page.get_text()

print(text[:2000])

Policy Usage Guide
In-patient Hospitalization
Expenses incurred for treatment that requires you to stay hospitalised for more
than24 hours. It includes expenses on – room charges, nursing expenses and 
intensive care unit charges, surgeon's fee, doctor's fee, anesthesia, blood, 
oxygen, operation theater charges, etc.
Day Care Treatment
Refers to treatment that requires less than 24 Hours of hospitalisation. We cover 
more than 540 such treatments. Some of the examples are Surgery for Cataract, 
Cancer Chemotherapy, Haemodialysis etc.
Pre & Post-Hospitalization Medical Expenses
Medical expenses incurred on doctor’s consultation fee, prescribed diagnostics 
and prescribed medicines 30 days before date of admission and 60 days after date 
of discharge.
Domiciliary Hospitalization
Medical expenses incurred during his/her treatment at home so long as it involves 
medical treatment for a period exceeding 3 days. T&C apply.
Automatic Recharge
Reinstates up to the amount of Sum Insured, once 

Loading All Insurance Documents

Here, I load all 7 insurance PDFs and extract text from every page.

For each page, I store:
- Extracted text
- Source PDF name
- Page number

Total pages loaded: 131

In [4]:
import fitz
import os

all_documents = []

for filename in pdf_files:
    pdf_path = os.path.join(pdf_folder, filename)
    
    doc = fitz.open(pdf_path)
    
    for page_number, page in enumerate(doc):
        page_text = page.get_text()
        
        all_documents.append({
            "text": page_text,
            "source": filename,
            "page": page_number + 1
        })

print("Total pages loaded:", len(all_documents))
print("Total PDFs:", len(pdf_files))

FileDataError: 'data\.ipynb_checkpoints' is no file

In [10]:
print(all_documents[0])

{'text': "Policy Usage Guide\nIn-patient Hospitalization\nExpenses incurred for treatment that requires you to stay hospitalised for more\nthan24 hours. It includes expenses on – room charges, nursing expenses and \nintensive care unit charges, surgeon's fee, doctor's fee, anesthesia, blood, \noxygen, operation theater charges, etc.\nDay Care Treatment\nRefers to treatment that requires less than 24 Hours of hospitalisation. We cover \nmore than 540 such treatments. Some of the examples are Surgery for Cataract, \nCancer Chemotherapy, Haemodialysis etc.\nPre & Post-Hospitalization Medical Expenses\nMedical expenses incurred on doctor’s consultation fee, prescribed diagnostics \nand prescribed medicines 30 days before date of admission and 60 days after date \nof discharge.\nDomiciliary Hospitalization\nMedical expenses incurred during his/her treatment at home so long as it involves \nmedical treatment for a period exceeding 3 days. T&C apply.\nAutomatic Recharge\nReinstates up to the 

Metadata provides additional information about each document.

For this project, I store:
- Source file name
- Page number
- Document type

In [11]:
document_types = {}

for filename in pdf_files:
    if "policy-usage-guide" in filename:
        document_type = "Policy Usage Guide"
    elif "brochure" in filename:
        document_type = "Brochure"
    elif "prospectus-cum-sales-literature" in filename:
        document_type = "Prospectus / Sales Literature"
    else:
        document_type = "Other"
    
    document_types[filename] = document_type

for filename, doc_type in document_types.items():
    print(filename, "->", doc_type)

.ipynb_checkpoints -> Other
care--policy-usage-guide.pdf -> Policy Usage Guide
care-freedom-(health-insurance-product)---brochure.pdf -> Brochure
care-freedom-(health-insurance-product)---prospectus-cum-sales-literature.pdf -> Prospectus / Sales Literature
care-opd---prospectus-cum-sales-literature.pdf -> Prospectus / Sales Literature
care-plus-(health-insurance-product)---brochure.pdf -> Brochure
care-plus--(health-insurance-product)---prospectus-cum-sales-literature.pdf -> Prospectus / Sales Literature
care-supreme----policy-usage-guide.pdf -> Policy Usage Guide


In [12]:
for document in all_documents:
    document["document_type"] = document_types[document["source"]]

print(all_documents[0])

NameError: name 'all_documents' is not defined

Text Cleaning

The extracted PDF text may contain unnecessary line breaks,
extra spaces, and formatting issues.

I clean the extracted text using regular expressions (`re`)
to make it more suitable for chunking and retrieval.

In [13]:
import re
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'(?i)(?<=\d)hours', ' hours', text)
    return text.strip()
for document in all_documents:
    document["clean_text"] = clean_text(document["text"])

print(all_documents[0]["clean_text"][:2000])

Policy Usage Guide In-patient Hospitalization Expenses incurred for treatment that requires you to stay hospitalised for more than24 hours. It includes expenses on – room charges, nursing expenses and intensive care unit charges, surgeon's fee, doctor's fee, anesthesia, blood, oxygen, operation theater charges, etc. Day Care Treatment Refers to treatment that requires less than 24 Hours of hospitalisation. We cover more than 540 such treatments. Some of the examples are Surgery for Cataract, Cancer Chemotherapy, Haemodialysis etc. Pre & Post-Hospitalization Medical Expenses Medical expenses incurred on doctor’s consultation fee, prescribed diagnostics and prescribed medicines 30 days before date of admission and 60 days after date of discharge. Domiciliary Hospitalization Medical expenses incurred during his/her treatment at home so long as it involves medical treatment for a period exceeding 3 days. T&C apply. Automatic Recharge Reinstates up to the amount of Sum Insured, once during 

In [14]:
import re

def clean_text(text):
    
    text = re.sub(r'\s+', ' ', text)

    # Fix missing space between words and numbers
    text = re.sub(r'([a-zA-Z])(\d)', r'\1 \2', text)
    text = re.sub(r'(\d)([a-zA-Z])', r'\1 \2', text)

    return text.strip()

for document in all_documents:
    document["clean_text"] = clean_text(document["text"])

print(all_documents[0]["clean_text"][:2000])

Policy Usage Guide In-patient Hospitalization Expenses incurred for treatment that requires you to stay hospitalised for more than 24 hours. It includes expenses on – room charges, nursing expenses and intensive care unit charges, surgeon's fee, doctor's fee, anesthesia, blood, oxygen, operation theater charges, etc. Day Care Treatment Refers to treatment that requires less than 24 Hours of hospitalisation. We cover more than 540 such treatments. Some of the examples are Surgery for Cataract, Cancer Chemotherapy, Haemodialysis etc. Pre & Post-Hospitalization Medical Expenses Medical expenses incurred on doctor’s consultation fee, prescribed diagnostics and prescribed medicines 30 days before date of admission and 60 days after date of discharge. Domiciliary Hospitalization Medical expenses incurred during his/her treatment at home so long as it involves medical treatment for a period exceeding 3 days. T&C apply. Automatic Recharge Reinstates up to the amount of Sum Insured, once during

In [16]:
!pip install langchain-text-splitters

   ---------------------------------------- 0.0/571.7 kB ? eta -:--:--
   ---------------------------------------- 571.7/571.7 kB 5.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/760.7 kB ? eta -:--:--
   ---------------------------------------- 760.7/760.7 kB 5.5 MB/s eta 0:00:00

   ----- ---------------------------------- 1/8 [websockets]
   ----- ---------------------------------- 1/8 [websockets]
   -------------------- ------------------- 4/8 [langchain-protocol]
   ------------------------- -------------- 5/8 [langsmith]
   ------------------------- -------------- 5/8 [langsmith]
   ------------------------- -------------- 5/8 [langsmith]
   ------------------------- -------------- 5/8 [langsmith]
   ------------------------- -------------- 5/8 [langsmith]
   ------------------------- -------------- 5/8 [langsmith]
   ------------------------- -------------- 5/8 [langsmith]
   ------------------------- -------------- 5/8 [langsmith]
   -----------------------

Text Chunking

Large documents are divided into smaller chunks so that the
retriever can efficiently search for relevant information.

I use RecursiveCharacterTextSplitter with:
- Chunk size: 512 characters
- Chunk overlap: 100 characters

The overlap helps reduce information loss when relevant text
falls near a chunk boundary.

Total chunks generated: 872

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=100
)
chunks = []
for document in all_documents:
    split_texts = text_splitter.split_text(document["clean_text"])
    
    for chunk in split_texts:
        chunks.append({
            "text": chunk,
            "source": document["source"],
            "page": document["page"],
            "document_type": document["document_type"]
        })

print("Total chunks:", len(chunks))
print("\nFirst chunk:")
print(chunks[0])

NameError: name 'all_documents' is not defined

## 7. Generating Embeddings

In this step, the document chunks are converted into numerical
vector representations using the `all-MiniLM-L6-v2` embedding model.

These embeddings help the system understand the semantic similarity
between the user's query and the insurance document chunks.

The generated embeddings will later be stored in ChromaDB for
efficient similarity-based retrieval.

In [1]:
!pip install sentence-transformers

   ---------------------------------------- 0.0/739.8 kB ? eta -:--:--
   -------------- ------------------------- 262.1/739.8 kB ? eta -:--:--
   ---------------------------------------- 739.8/739.8 kB 2.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/796.8 kB ? eta -:--:--
   ------------- -------------------------- 262.1/796.8 kB ? eta -:--:--
   ---------------------------------------- 796.8/796.8 kB 3.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.0 MB 1.9 MB/s eta 0:00:02
   ---------- ----------------------------- 1.0/4.0 MB 2.4 MB/s eta 0:00:02
   ------------ --------------------------- 1.3/4.0 MB 2.0 MB/s eta 0:00:02
   --------------- ------------------------ 1.6/4.0 MB 1.9 MB/s eta 0:00:02
   ------------------ --------------------- 1.8/4.0 MB 1.8 MB/s eta 0:00:02
   -------------------- ------------------- 2.1/4.0 MB 1.7 MB/s eta 0:00:02
   ------------------------

In [2]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\LENOVO 4HIN\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LENOVO 4HIN\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!
